In [1]:
import pandas as pd
import numpy as np
import sqlite3

import sys
import os

sys.path.append(os.path.abspath(".."))
import app.features.features as features
import catboost

In [ ]:
conn = sqlite3.connect("../data/riot_lol.sqlite")

# "PRAGMA table_info(player_ranks);"

query = """
SELECT
    p.match_id,
    p.team_id,
    p.puuid,
    p.win,
    p.champion_id,
    p.team_position,
    r.tier,
    r.division,
    r.league_points,
    r.wins,
    r.losses,
    m.game_creation
FROM participants p
LEFT JOIN matches m
    ON p.match_id = m.match_id
LEFT JOIN player_ranks r
    ON r.puuid = p.puuid
    AND r.snapshot_ts = (
        SELECT MAX(r2.snapshot_ts)
        FROM player_ranks r2
        WHERE r2.puuid = p.puuid
        AND r2.snapshot_ts <= m.game_creation
    )
"""

df = pd.read_sql_query(query, conn)

# print(df.columns)
# print(df.columns[df.columns == 'league_points'])

df.dropna(inplace = True)

rf_df = features.build_features(df)
rf_df = features.encoding_categoricals(rf_df)
print(df.head(5))

         match_id  team_id                                              puuid  \
0  NA1_5494453720      100  BkoKffZBprI9447fyoRLg9qYANCXTGlcZqqcoLC2uLPwZO...   
1  NA1_5494453720      100  QJ04p4VcgKybTJEEtlUbNInOnEg0Cb1Rv5CPhbNR807ifA...   
2  NA1_5494453720      100  0BUa_yL5ZEg6Y_rCSQd95pB8y7MAEThgSNLZd8jtprMdHO...   
3  NA1_5494453720      100  7orc7p6ptbGwN4j1ibBz9beGiKdqfJ4qCvY6H0v7p9cWUk...   
4  NA1_5494453720      100  mPlS9w6hsowbBuJrzTmmKuSeEyAkl2DXetSvXMn5aVpn4J...   

   win  champion_id team_position         tier division  league_points   wins  \
0    1          126           TOP   CHALLENGER        I         2477.0  293.0   
1    1          238        JUNGLE  GRANDMASTER        I          834.0   72.0   
2    1          112        MIDDLE  GRANDMASTER        I          886.0  135.0   
3    1           81        BOTTOM   CHALLENGER        I         2205.0  125.0   
4    1           34       UTILITY  GRANDMASTER        I          769.0  125.0   

   losses  game_creation  

In [2]:
from app.inference import Predictor

predictor = Predictor()
predictor.predict_live("Evolev420", "1678")

REQUEST: https://americas.api.riotgames.com/riot/account/v1/accounts/by-riot-id/Evolev420/1678
STATUS: 200
RESPONSE: {"puuid":"jW9gqPy9FtqwTa6GMNi_lLX3eOW61h3B1ar-5-EQHX3gIm1qd2rLXM-ZH0mh-ggbRYh2BrgChnjc_Q","gameName":"Evolev420","tagLine":"1678"}
dict_keys(['gameId', 'mapId', 'gameMode', 'gameType', 'gameQueueConfigId', 'participants', 'observers', 'platformId', 'bannedChampions', 'gameStartTime', 'gameLength'])
<class 'list'>


{'team_100_win_prob': 0.27676560337751693,
 'team_200_win_prob': 0.7410546907338038}